# Init

In [9]:
%matplotlib qt
import numpy as np
import scipy.constants as phy_const
import matplotlib.pyplot as plt
import pickle

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import os
import pandas as pd
import pickle
import glob
import sys
import configparser
from tqdm import tqdm

from cycler import cycler
import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.pyplot import cm
from pylab import arange, pi, sin, cos, sqrt
from matplotlib import rcParams
import matplotlib.animation as animation
from scipy import constants as cons
from matplotlib import gridspec
import matplotlib.colors as mpl_colors
from matplotlib.widgets import Slider, TextBox, Button, RadioButtons
from matplotlib import ticker
from matplotlib.path import Path
from mpl_toolkits.axes_grid1.inset_locator import mark_inset
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.ticker import FormatStrFormatter
from matplotlib.colors import Normalize, BoundaryNorm, LogNorm
from matplotlib.ticker import MaxNLocator
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.colors import ListedColormap
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as patches
import matplotlib.lines as mlines


def find_nearest(array, value):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return idx

# Main style
plt.style.use('classic')

# Alejandro parameters
plt.rcParams["font.family"]         = 'Times New Roman'
plt.rcParams["font.weight"]         = 'normal'
plt.rcParams['figure.facecolor']    = 'white' 
plt.rcParams["font.size"]           = 12
plt.rcParams["lines.linewidth"]     = 2

# Grid parameters
plt.rcParams['axes.grid'] = True          
plt.rcParams['grid.color'] = '0.85'       
plt.rcParams['grid.linestyle'] = '-'     
plt.rcParams['grid.linewidth'] = 0.7      
plt.rcParams['grid.alpha'] = 0.7    
plt.rcParams['axes.grid.axis'] = 'both' 
plt.rcParams['axes.grid.which'] = 'major'
plt.rcParams['axes.axisbelow'] = True

# Choices of colors cycler
dashes = [( ), (5, 3), (2, 2), (6, 2, 2, 2)]  # solid, dashed, dotted, dash-dot
plt.rcParams['axes.prop_cycle'] = cycler('color', ['k', 'r', 'b', 'g'])# + cycler('ls', ['-', '--', ':', '-.']) + cycler('dashes', dashes)

# Ticks limits
plt.rcParams.update({
    'axes.autolimit_mode': 'round_numbers',  # keeps tick limits tidy
    'axes.xmargin': 0.0,  # no extra margin added
    'axes.ymargin': 0.0,
})

# Legend
plt.rcParams.update({
    'legend.loc': 'best',            # Auto place; or 'upper right', etc.
    'legend.frameon': False,         # No frame
    'legend.fontsize': 12,            # Smaller font size
    'legend.borderaxespad': 0.5,     # Padding between legend and axes
    'legend.labelspacing': 0.01,      # Vertical space between entries
    'legend.handletextpad': 0.3,     # Space between line and text
    'legend.columnspacing': 1.0,     # Horizontal space between columns
    'legend.numpoints': 1,           # One point per line symbol
    'legend.fancybox': False,        # No rounded box
    'legend.handlelength': 1,  # length of the legend line
    'legend.handleheight': 0.7,  # height of the legend handle (marker size)
})

plt.rcParams.update({
    'savefig.dpi': 300,
})


# Load config

In [2]:
RESULTSDIR = './Results/test_001/'
configFile = RESULTSDIR+'/Configuration.cfg'

ResultsFigs = RESULTSDIR+"/Figs"
ResultsData = RESULTSDIR+"/Data"

if not os.path.exists(ResultsFigs):
    os.makedirs(ResultsFigs)

In [3]:
config = configparser.ConfigParser()
config.read(configFile)

physicalParameters = config["Physical Parameters"]

VG       = float(physicalParameters["Gas velocity"])              # Gas velocity
M        = float(physicalParameters["Ion Mass"]) * phy_const.m_u  # Ion Mass
m        = phy_const.m_e                                          # Electron mass
R1       = float(physicalParameters["Inner radius"])              # Inner radius of the thruster
R2       = float(physicalParameters["Outer radius"])              # Outer radius of the thruster
A0       = np.pi * (R2**2 - R1**2)                                # Area of the thruster
LENGTH   = float(physicalParameters["Length of axis"])            # length of Axis of the simulation
L0       = float(physicalParameters["Length of thruster"])            # length of thruster (position of B_max)
alpha_B1 = float(physicalParameters["Anomalous transport alpha_B1"])  # Anomalous transport
alpha_B2 = float(physicalParameters["Anomalous transport alpha_B2"])  # Anomalous transport
mdot     = float(physicalParameters["Mass flow"])                     # Mass flow rate of propellant
Te_Cath  = float(physicalParameters["Temperature Cathode"])           # Electron temperature at the cathode
NI0      = float(physicalParameters["Initial plasma density"])  
TE0      = float(physicalParameters["Initial Temperature"])  
Rext     = float(physicalParameters["Ballast resistor"])              # Resistor of the ballast
V        = float(physicalParameters["Voltage"])                       # Potential difference
Circuit  = bool(config.getboolean("Physical Parameters", "Circuit", fallback=False))  # RLC Circuit
Estar    = float(physicalParameters["Crossover energy"])  # Crossover energy

# Magnetic field configuration
MagneticFieldConfig = config["Magnetic field configuration"]

if MagneticFieldConfig["Type"] == "Default":
    print(MagneticFieldConfig["Type"] + " Magnetic Field")

    Bmax       = float(MagneticFieldConfig["Max B-field"])       # Max Mag field
    LB1        = float(MagneticFieldConfig["Length B-field 1"])  # Length for magnetic field
    LB2        = float(MagneticFieldConfig["Length B-field 2"])  # Length for magnetic field
    saveBField = bool(config.getboolean("Magnetic field configuration", "Save B-field", fallback=False))

elif MagneticFieldConfig["Type"] == "StationaryCodeBField":
    print(MagneticFieldConfig["Type"] + " Magnetic Field")

    Bmax       = float(MagneticFieldConfig["Max B-field"])  # Max Mag field
    CmagIn     = float(MagneticFieldConfig["Cmag In"])      # Length for magnetic field
    CmagOut    = float(MagneticFieldConfig["Cmag Out"])     # Length for magnetic field
    saveBField = bool(config.getboolean("Magnetic field configuration", "Save B-field", fallback=False))

##########################################################
#           NUMERICAL PARAMETERS
##########################################################
NumericsConfig = config["Numerical Parameteres"]

NBPOINTS   = int(NumericsConfig["Number of points"])    # Number of cells
SAVERATE   = int(NumericsConfig["Save rate"])           # Rate at which we store the data
CFL        = float(NumericsConfig["CFL"])               # Nondimensional size of the time step
TIMEFINAL  = float(NumericsConfig["Final time"])        # Last time of simulation
Results    = NumericsConfig["Result dir"]               # Name of result directory
TIMESCHEME = NumericsConfig["Time integration"]         # Time integration scheme


StationaryCodeBField Magnetic Field


# Load pickle

In [21]:
files = sorted(glob.glob(ResultsData + "/*.pkl"), key=os.path.getmtime)
data = {k: [] for k in
        ["time","ng","ni","ui","Te","ve","P_inlet","P_outlet",
         "Current","Voltage","MagneticB","x_center"]}

for fpath in tqdm(files):
    t, P, U, Pin, Pout, J, V, B, xc = pickle.load(open(fpath, "rb"))

    data["time"].append(t)
    data["ng"].append(P[0]);  data["ni"].append(P[1])
    data["ui"].append(P[2]);  data["Te"].append(P[3])
    data["ve"].append(P[4])

    data["P_inlet"].append(Pin)
    data["P_outlet"].append(Pout)
    data["Current"].append(J)
    data["Voltage"].append(V)
    data["MagneticB"].append(B)
    data["x_center"].append(xc)

# convert to numpy arrays
my_dic = {k: np.array(v) for k, v in data.items()}


100%|██████████| 8413/8413 [00:00<00:00, 41552.31it/s]


# Profiles

## Current evolution

In [19]:
fig = plt.figure(figsize=(3.5, 2.5), dpi=250)
gs1 = gridspec.GridSpec(1, 1)
gs1.update(hspace=0.2, wspace=0.1, left=0.12, right=0.95, bottom=0.18, top=0.95)

ax1 = plt.subplot(gs1[0, 0])
ax1.set_ylabel(r'Current (A)', fontsize=12)
ax1.set_xlabel(r'$t$ (µs)', fontsize=12, labelpad=3.25)
ax1.set_xlim(0, 2000)
ax1.set_ylim(0, 150)
ax1.tick_params(axis='both', labelsize=10)
ax1.yaxis.set_major_locator(MaxNLocator(nbins=4))
ax1.xaxis.set_major_locator(MaxNLocator(nbins=4))

time = my_dic["time"]*1e6  # in ms
current = my_dic["Current"]
ax1.plot(time, current)

## Neutral 2D

In [28]:
fig = plt.figure(figsize=(3.5, 2.5), dpi=250)
gs1 = gridspec.GridSpec(1, 1)
gs1.update(hspace=0.2, wspace=0.1, left=0.12, right=0.95, bottom=0.18, top=0.95)

ax1 = plt.subplot(gs1[0, 0])
ax1.set_ylabel(r'$x$ (cm)', fontsize=12)
ax1.set_xlabel(r'$t$ (µs)', fontsize=12, labelpad=3.25)
ax1.set_xlim(1500, 2000)
ax1.set_ylim(0, 2.5)
ax1.tick_params(axis='both', labelsize=10)
ax1.yaxis.set_major_locator(MaxNLocator(nbins=4))
ax1.xaxis.set_major_locator(MaxNLocator(nbins=4))

time = my_dic["time"]*1e6  # in ms
length = my_dic["x_center"][0]*100  # in cm
ng = my_dic["ng"]*1E-18
ax1.pcolormesh(time, length, ng.T, vmin=5, vmax=100, shading='auto', cmap='nipy_spectral')

## Temperature profile

In [32]:
fig = plt.figure(figsize=(3.5, 2.5), dpi=250)
gs1 = gridspec.GridSpec(1, 1)
gs1.update(hspace=0.2, wspace=0.1, left=0.12, right=0.95, bottom=0.18, top=0.95)

ax1 = plt.subplot(gs1[0, 0])
ax1.set_ylabel(r'$x$ (cm)', fontsize=12)
ax1.set_xlabel(r'$t$ (µs)', fontsize=12, labelpad=3.25)
ax1.set_xlim(0, 2.5)
ax1.set_ylim(0, 50)
ax1.tick_params(axis='both', labelsize=10)
ax1.yaxis.set_major_locator(MaxNLocator(nbins=4))
ax1.xaxis.set_major_locator(MaxNLocator(nbins=4))

time = my_dic["time"]*1e6  # in ms
length = my_dic["x_center"][0]*100  # in cm
Te = my_dic["Te"]

t = [1500, 1550, 1600, 1650]

for ti in t:
    idx = find_nearest(time, ti)
    ax1.plot(length, Te[idx], label=f'{ti} µs')

## ni profile

In [44]:
fig = plt.figure(figsize=(3.5, 2.5), dpi=250)
gs1 = gridspec.GridSpec(1, 1)
gs1.update(hspace=0.2, wspace=0.1, left=0.18, right=0.95, bottom=0.18, top=0.95)

ax1 = plt.subplot(gs1[0, 0])
ax1.set_ylabel(r'$n_\mathrm{i}$ (m$^{-3}$)', fontsize=12)
ax1.set_xlabel(r'$x$ (cm)', fontsize=12, labelpad=3.25)
ax1.set_xlim(0, 2.5)
ax1.set_ylim(0, 1)
ax1.tick_params(axis='both', labelsize=10)
ax1.yaxis.set_major_locator(MaxNLocator(nbins=4))
ax1.xaxis.set_major_locator(MaxNLocator(nbins=4))

time = my_dic["time"]*1e6  # in ms
length = my_dic["x_center"][0]*100  # in cm
ni = my_dic["ni"]*1E-17

t = [1500, 1550, 1600, 1650]

for ti in t:
    idx = find_nearest(time, ti)
    ax1.plot(length, ni[idx], label=f'{ti} µs')

## Magnetic profile

In [49]:
fig = plt.figure(figsize=(3.5, 2.5), dpi=250)
gs1 = gridspec.GridSpec(1, 1)
gs1.update(hspace=0.2, wspace=0.1, left=0.18, right=0.95, bottom=0.18, top=0.95)

ax1 = plt.subplot(gs1[0, 0])
ax1.set_ylabel(r'$B$ (G)', fontsize=12)
ax1.set_xlabel(r'$x$ (cm)', fontsize=12, labelpad=3.25)
ax1.set_xlim(0, 2.5)
ax1.set_ylim(0, 210)
ax1.tick_params(axis='both', labelsize=10)
ax1.yaxis.set_major_locator(MaxNLocator(nbins=4))
ax1.xaxis.set_major_locator(MaxNLocator(nbins=4))

time = my_dic["time"]*1e6  # in ms
length = my_dic["x_center"][0]*100  # in cm
BB = my_dic["MagneticB"][0]*1E4  # in G

ax1.plot(length, BB, label=f'{ti} µs')